In [1]:
import numpy as np
import scipy
import cdd
import itertools
import cupy as cp
import time

In [2]:
# Define the hypercube
vertices = np.array(list(itertools.product([0, 1], repeat=3)))
centerd_vertices = vertices - 0.5
print(centerd_vertices)

[[-0.5 -0.5 -0.5]
 [-0.5 -0.5  0.5]
 [-0.5  0.5 -0.5]
 [-0.5  0.5  0.5]
 [ 0.5 -0.5 -0.5]
 [ 0.5 -0.5  0.5]
 [ 0.5  0.5 -0.5]
 [ 0.5  0.5  0.5]]


In [3]:
# Set up constraint matrix for hypercube in V representation
_, num_dims = np.shape(vertices)
t = np.ones(len(vertices))
hypercube_mat = cdd.matrix_from_array(
    np.column_stack((t.T, centerd_vertices)), rep_type=cdd.RepType.GENERATOR
)

# Can obtain the V and H representations easily
hypercube = cdd.polyhedron_from_matrix(hypercube_mat)
hypercube_inequalities = cdd.copy_inequalities(hypercube).array

print(cdd.copy_generators(hypercube))
print(cdd.copy_inequalities(hypercube))


V-representation
begin
 8 4 real
  1 -5.000000000E-01 -5.000000000E-01 -5.000000000E-01
  1 -5.000000000E-01 -5.000000000E-01  5.000000000E-01
  1 -5.000000000E-01  5.000000000E-01 -5.000000000E-01
  1 -5.000000000E-01  5.000000000E-01  5.000000000E-01
  1  5.000000000E-01 -5.000000000E-01 -5.000000000E-01
  1  5.000000000E-01 -5.000000000E-01  5.000000000E-01
  1  5.000000000E-01  5.000000000E-01 -5.000000000E-01
  1  5.000000000E-01  5.000000000E-01  5.000000000E-01
end
H-representation
begin
 6 4 real
  1  0  0  2
  1  2  0  0
  1  0  2  0
  1  0  0 -2
  1  0 -2  0
  1 -2  0  0
end


In [4]:
# Define the intersecting planes
hyperplane_inequalities = np.array([[0, 0,1,0], [0, 0,0, 1]])
lin_set = set([0, 1])
augmented_inequalities = np.vstack(
    (hyperplane_inequalities, hypercube_inequalities)
)

intersection_mat = cdd.matrix_from_array(
    augmented_inequalities, rep_type=cdd.RepType.INEQUALITY, lin_set=lin_set
)
intersection = cdd.polyhedron_from_matrix(intersection_mat)
intersection_vertices = np.array(cdd.copy_generators(intersection).array)

print(cdd.copy_generators(intersection))
print(intersection_vertices)

V-representation
begin
 2 4 real
  1  5.000000000E-01  0  0
  1 -5.000000000E-01  0  0
end
[[ 1.   0.5  0.   0. ]
 [ 1.  -0.5  0.   0. ]]


In [5]:
def get_intersection_vertices(
    vertices: np.ndarray | cp.ndarray,
    correlation_signature: list[int] | None = None,
) -> np.ndarray | cp.ndarray:
    """Compute the intersection of the polytope with given vertices with
    hyperplanes as defined by the memory effect condition described by the
    given correlation signature."""
    xp = cp.get_array_module(vertices)
    # Set up cdd matrix object for the initial polytope
    t = xp.ones(len(vertices))
    polytope_mat = cdd.matrix_from_array(
        xp.column_stack((t.T, vertices)), rep_type=cdd.RepType.GENERATOR
    )

    # Get the halfspace representation inequalities
    polytope = cdd.polyhedron_from_matrix(polytope_mat)
    polytope_inequalities = xp.array(cdd.copy_inequalities(polytope).array)

    # Intersect the polytope with the hyperplanes
    if correlation_signature is None:
        correlation_signature = [1, -1, -1, 1]
    a, b, c, d = correlation_signature
    hyperplane_equations = xp.array(
        [[0, a, 0, b, 0, c, 0, d, 0], [0, 0, a, 0, b, 0, c, 0, d]]
    )
    lin_set = set([0, 1])
    augmented_inequalities = xp.vstack(
        (hyperplane_equations, polytope_inequalities)
    )
    intersection_mat = cdd.matrix_from_array(
        augmented_inequalities,
        rep_type=cdd.RepType.INEQUALITY,
        lin_set=lin_set,
    )
    intersection = cdd.polyhedron_from_matrix(intersection_mat)
    intersection_vertices = xp.array(cdd.copy_generators(intersection).array)

    # Truncate the
    truncated_vertices = intersection_vertices[:, 1:7]
    return truncated_vertices
